In [ ]:
# ============================================================
# Notebook 05 - Phase 3: CNN + BiLSTM Hybrid
# CS-419 Deep Learning Project - Speech Emotion Recognition
# ============================================================
# Experiments:
#   A. CNN-BiLSTM baseline
#   B. CNN-BiLSTM vs CNN-GRU
#   C. CNN-Attention (MultiHeadAttention)
#   D. Final best model comparison (MLP vs CNN vs CNN-LSTM)
# ============================================================

import sys
import os
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("../src")

from models import build_cnn_lstm, build_cnn_attention
from train import train_model, plot_history, compare_histories
from evaluate import evaluate_model, plot_model_comparison_bar
from data_loader import get_class_weights, load_tess_paths, split_dataset
from utils import set_seeds, save_results_csv
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

set_seeds(42)
os.makedirs("results/plots", exist_ok=True)
os.makedirs("results/models", exist_ok=True)

# ---- Cell 1: Load data ----

print("Loading cached spectrograms...")
train_data = np.load("results/cache/train_spec.npz")
val_data   = np.load("results/cache/val_spec.npz")
test_data  = np.load("results/cache/test_spec.npz")

X_train, y_train = train_data["X"], train_data["y"]
X_val,   y_val   = val_data["X"],   val_data["y"]
X_test,  y_test  = test_data["X"],  test_data["y"]

INPUT_SHAPE = X_train.shape[1:]

DATA_ROOT = "data/TESS Toronto emotional speech set data"
df = load_tess_paths(DATA_ROOT)
df_train, _, _ = split_dataset(df)
class_weights = get_class_weights(df_train)

# ---- Cell 2: Experiment A - CNN + BiLSTM ----

print("\n=== Experiment A: CNN + BiLSTM ===")
model_bilstm = build_cnn_lstm(
    input_shape=INPUT_SHAPE,
    cnn_filters=(32, 64, 128),
    lstm_units=128,
    dropout_rate=0.4,
    l2_reg=1e-4,
    optimizer="adam",
    learning_rate=1e-3,
)
model_bilstm.summary()

hist_bilstm = train_model(
    model_bilstm, X_train, y_train, X_val, y_val,
    model_name="CNN_BiLSTM",
    epochs=60, batch_size=32,
    class_weights=class_weights,
)
plot_history(hist_bilstm, "CNN_BiLSTM")
res_bilstm = evaluate_model(model_bilstm, X_test, y_test, model_name="CNN_BiLSTM")

# ---- Cell 3: Experiment B - CNN + GRU ----

print("\n=== Experiment B: CNN + BiGRU ===")

def build_cnn_bigru(input_shape, gru_units=128):
    inp = keras.Input(shape=input_shape)
    x = inp
    for i, filters in enumerate([32, 64, 128]):
        x = layers.Conv2D(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D(pool_size=(2, 1))(x)

    sh = x.shape
    seq_len = sh[2]
    feat_dim = sh[1] * sh[3]
    x = layers.Reshape((seq_len, feat_dim))(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Bidirectional(
        layers.GRU(gru_units, return_sequences=False, dropout=0.3)
    )(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(7, activation="softmax")(x)

    model = keras.Model(inp, out, name="CNN_BiGRU")
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

model_bigru = build_cnn_bigru(INPUT_SHAPE, gru_units=128)
hist_bigru = train_model(model_bigru, X_train, y_train, X_val, y_val,
                          model_name="CNN_BiGRU", epochs=60, batch_size=32,
                          class_weights=class_weights)
res_bigru = evaluate_model(model_bigru, X_test, y_test, model_name="CNN_BiGRU")

compare_histories(
    {"CNN-BiLSTM": hist_bilstm, "CNN-BiGRU": hist_bigru},
    metric="val_accuracy"
)

# ---- Cell 4: Experiment C - CNN + Attention ----

print("\n=== Experiment C: CNN + Multi-Head Attention ===")
model_attn = build_cnn_attention(
    input_shape=INPUT_SHAPE,
    cnn_filters=(32, 64, 128),
    dropout_rate=0.4,
    optimizer="adam",
    learning_rate=1e-3,
)
hist_attn = train_model(model_attn, X_train, y_train, X_val, y_val,
                         model_name="CNN_Attention", epochs=60, batch_size=32,
                         class_weights=class_weights)
res_attn = evaluate_model(model_attn, X_test, y_test, model_name="CNN_Attention")

# ---- Cell 5: Final all-models comparison ----

print("\n=== Final Comparison: All Models ===")

# Load MLP best model results
mlp_results = np.load("results/cache/test_flat.npz")
mlp_model   = keras.models.load_model("results/models/mlp_best.keras")
res_mlp     = evaluate_model(mlp_model,
                              mlp_results["X"], mlp_results["y"],
                              model_name="MLP_best")

cnn_model = keras.models.load_model("results/models/cnn_best.keras")
res_cnn   = evaluate_model(cnn_model, X_test, y_test, model_name="CNN_best")

all_summary = [
    {"model": "MLP (baseline)",    "accuracy": res_mlp["accuracy"],    "macro_f1": res_mlp["macro_f1"]},
    {"model": "CNN (4 blocks)",    "accuracy": res_cnn["accuracy"],    "macro_f1": res_cnn["macro_f1"]},
    {"model": "CNN + BiLSTM",      "accuracy": res_bilstm["accuracy"], "macro_f1": res_bilstm["macro_f1"]},
    {"model": "CNN + BiGRU",       "accuracy": res_bigru["accuracy"],  "macro_f1": res_bigru["macro_f1"]},
    {"model": "CNN + Attention",   "accuracy": res_attn["accuracy"],   "macro_f1": res_attn["macro_f1"]},
]
df_all = save_results_csv(all_summary, "results/all_model_results.csv")
print("\nFinal Results:")
print(df_all.to_string(index=False))

plot_model_comparison_bar(all_summary)

# Save best model
best_row = max(all_summary, key=lambda x: x["accuracy"])
print(f"\nBest model: {best_row['model']} with {best_row['accuracy']*100:.2f}% accuracy")

if best_row["model"] == "CNN + BiLSTM":
    model_bilstm.save("results/models/best_overall.keras")
elif best_row["model"] == "CNN + BiGRU":
    model_bigru.save("results/models/best_overall.keras")
elif best_row["model"] == "CNN + Attention":
    model_attn.save("results/models/best_overall.keras")
else:
    cnn_model.save("results/models/best_overall.keras")

print("Best model saved to results/models/best_overall.keras")
print("\n=== Phase 3 Complete ===")